In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from arch.unitroot import PhillipsPerron
from scipy.stats import kurtosis, skew
from statsmodels.tsa.vector_ar.vecm import coint_johansen


### Methodology:
1. Obtain the serie of prices of our assets
2. transform the prices (ex: log)
3. Selecting the pairs that interesting us
3. compute rolling mean + rolling standard deviation of the spreads
4. Compute the Z score
5. Generate trading signal
6. execute or no the trade
7. calcul of the performance metrics

In [50]:
df_prices = pd.read_csv("data.csv")

On transforme d'abord les prix pour obtenir des log prices

In [51]:
log_prices = np.log(df_prices.iloc[:, 1:])
log_returns = log_prices.diff()

### Cointegration tests

First recall the definition of cointegration from the course

Cointegration: Two or more non-stationary series share a common
stochastic trend. A linear combination of them is stationary.

Statistical Interpretation: Deviation from equilibrium are mean-reverting


### Testing for Cointegration: The Johansen Procedure

##### Step 1: Confirm both series are I(1) (unit root tests)

That means that the serie of price is not stationnary but its increment process are. For that purpose we can use Augmented Dickey-Fuller (ADF) or Phillips-Perron (PP) test.

In [52]:
def compute_stats(data):

    results = []

    for col in data.columns:
        
        series = data[col].dropna()

        adf_p = adfuller(series)[1]
        pp_p = PhillipsPerron(series).pvalue
        
        mean = series.mean()
        std = series.std()
        min_v = series.min()
        max_v = series.max()
        kurt = kurtosis(series)
        skewness = skew(series)

        results.append([col,round(adf_p,3),round(pp_p,3),round(mean,3),round(std,3),round(min_v,3),round(max_v,3),round(kurt,3),round(skewness,3)])

    columns = ["Symbol","ADF test","PP test","Mean","Std","Min","Max","Kurtosis","Skewness"]

    return pd.DataFrame(results, columns=columns)

levels_table = compute_stats(log_prices)
returns_table = compute_stats(log_returns)

print("LEVELS")
levels_table

LEVELS


,Symbol,ADF test,PP test,Mean,Std,Min,Max,Kurtosis,Skewness
0,1688.T,0.712,0.720,6.130,0.280,5.591,6.636,-1.274,-0.079
1,ADM,0.518,0.512,3.909,0.301,3.206,4.477,-1.207,-0.032
2,AGRO,0.273,0.221,2.021,0.275,1.150,2.464,0.477,-0.875
3,BG,0.794,0.787,4.218,0.340,3.235,4.816,-0.837,-0.532
4,CF,0.782,0.809,3.999,0.428,2.906,4.752,-1.275,-0.293
5,CLSEL.NS,0.784,0.740,5.213,0.438,4.411,6.051,-1.327,-0.104
6,DBA,0.966,0.969,2.875,0.229,2.452,3.310,-0.966,0.276
7,FDP,0.391,0.221,3.280,0.191,2.884,3.773,-0.451,0.611
8,KOHINOOR.NS,0.312,0.407,3.174,0.796,1.558,4.811,-1.263,-0.391
9,KRBL.NS,0.019,0.018,5.678,0.267,4.534,6.382,0.054,-0.134


In [53]:
print("\nLOG RETURNS")
returns_table


LOG RETURNS


,Symbol,ADF test,PP test,Mean,Std,Min,Max,Kurtosis,Skewness
0,1688.T,0.0,0.0,0.000,0.016,-0.136,0.213,20.121,0.954
1,ADM,0.0,0.0,0.000,0.018,-0.277,0.098,32.436,-2.259
2,AGRO,0.0,0.0,0.000,0.026,-0.261,0.126,10.371,-1.020
3,BG,0.0,0.0,0.000,0.020,-0.154,0.135,7.161,-0.438
4,CF,0.0,0.0,0.001,0.025,-0.167,0.151,4.452,-0.358
5,CLSEL.NS,0.0,0.0,0.001,0.028,-0.146,0.151,4.067,0.635
6,DBA,0.0,0.0,0.000,0.008,-0.050,0.030,2.138,-0.333
7,FDP,0.0,0.0,0.000,0.023,-0.172,0.225,16.931,0.296
8,KOHINOOR.NS,0.0,0.0,-0.001,0.030,-0.219,0.182,6.749,0.826
9,KRBL.NS,0.0,0.0,-0.000,0.031,-0.223,0.178,9.225,-0.487


These table, very similar to the one exposed in the main article of this trading game "Trading Games: Beating Passive Strategies in the Bullish
Crypto Market" confirm that any prices are stationnary with high p-values for ADF test and PP test and every log returns are stationnary with p-value < 0.001. We can thus keep all the series of prices that seems to be I(1).

Step 2: Test for cointegration using Johansen test

In [54]:
log_prices

,1688.T,ADM,AGRO,BG,CF,CLSEL.NS,DBA,FDP,KOHINOOR.NS,KRBL.NS,LTFOODS.NS,MOS,NTR,RJA,VC2.SI,VFF,Futures
0,5.978886,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.316154,6.322520,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5.978886,3.447051,2.219076,3.969658,3.557433,NaN,2.791563,3.671175,4.298645,6.319858,NaN,3.154218,3.735951,1.801710,0.285420,NaN,NaN
2,5.978886,3.439288,2.240130,3.990905,3.557665,NaN,2.791035,3.676912,4.329417,6.328236,NaN,3.145172,3.747031,1.805005,0.270605,NaN,NaN
3,5.988961,3.455991,2.235384,4.005485,3.568039,NaN,2.789452,3.697669,4.381401,6.339081,NaN,3.160577,3.750457,1.805005,0.285420,NaN,NaN
4,5.966147,3.449293,2.220043,3.997935,3.569644,NaN,2.784155,3.697877,4.480174,6.358023,NaN,3.162067,3.753692,1.798404,0.285420,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2133,6.314815,4.242908,2.232163,4.792562,4.647271,5.560682,3.257712,3.754595,3.199489,5.800607,5.940697,3.314550,4.327306,NaN,-0.133531,1.238374,2.362739
2134,6.306275,4.219361,2.211566,4.758921,4.666265,NaN,3.258865,3.744314,NaN,NaN,NaN,3.287655,4.309725,NaN,-0.139262,1.205971,2.359910
2135,6.310827,4.207971,2.260721,4.745019,4.651195,5.512218,3.262701,3.753262,3.177637,5.769882,5.906859,3.258481,4.298509,NaN,-0.174353,1.232560,2.345645
2136,6.302619,4.197653,2.272126,4.729156,4.707546,5.553347,3.273743,3.749504,3.221672,5.809493,6.064134,3.268808,4.309322,NaN,-0.168419,1.217876,2.378620


In [62]:
assets = log_prices.columns
results = []

for i in range(len(assets)):
    
    for j in range(i+1, len(assets)):
        
        a = assets[i]
        b = assets[j]

        

        pair_df = log_prices[[a, b]].dropna()

        if len(pair_df) < 50:
            continue
        joh = coint_johansen(pair_df,det_order=0,k_ar_diff=1)

        trace_stat = joh.lr1[0]
        crit_95 = joh.cvt[0,1]

        cointegrated = trace_stat > crit_95
        results.append({"Asset 1": a,"Asset 2": b,"Trace Stat": trace_stat,"Critical 95%": crit_95,"Cointegrated": cointegrated})

results_df = pd.DataFrame(results)

results_df = results_df.sort_values("Trace Stat", ascending=False)

results_df.head(30)

,Asset 1,Asset 2,Trace Stat,Critical 95%,Cointegrated
100,KOHINOOR.NS,KRBL.NS,29.483095,15.4943,True
7,1688.T,KOHINOOR.NS,26.314300,15.4943,True
0,1688.T,ADM,23.716654,15.4943,True
104,KOHINOOR.NS,RJA,23.310490,15.4943,True
12,1688.T,RJA,23.183869,15.4943,True
31,AGRO,BG,21.300657,15.4943,True
92,FDP,KRBL.NS,19.969297,15.4943,True
22,ADM,KOHINOOR.NS,19.634024,15.4943,True
102,KOHINOOR.NS,MOS,19.306732,15.4943,True
109,KRBL.NS,MOS,19.018976,15.4943,True


### Step 3: If cointegrated, estimate the relationship

For each cointegrated pairs there exist a cointegrated relationship of the following form:

$$
Y_{1,t} = \mu + \gamma Y_{2,t} + \varepsilon_t, \quad \varepsilon_t \sim I(0)
$$

where $\mu$ is the intercept,$\gamma$ is the cointegrating coefficient,and $\varepsilon_t$
is a stationary error term. To estimate we can use VECM estimation like in the course, but this method is less direct to obtain $\mu$ and $\gamma$, in the main paper they employ the Eagle and Granger method which is way more adapted in our situation.

In [ ]:
#We keep only the cointegrated pairs

Asset_cointegrated = results_df[results_df["Cointegrated"]== 1][["Asset 1", "Asset 2"]]

#This function aim to estimate the mu and gamma coefficient for a cointegrated pair
def estimate_coint_coeffs(Y):

    Y = Y.dropna().values
    Asset_1 = Y[:, 0]
    Asset_2 = Y[:, 1]

    X = np.column_stack([np.ones(len(Asset_2)), Asset_2])

    beta = np.linalg.lstsq(X, Asset_1, rcond=None)[0]

    mu = beta[0]
    gamma = beta[1]

    return mu, gamma


Afet that we can write the spread $S_t$:

$$
S_t = Y_{1,t} - \gamma Y_{2,t}
$$

and thus this spread is mean reverting and can be used for a trading strategy based on statistical arbitrage


To generate trading signals, we normalize the spread by calculating the Z-score, which measures the spread's deviation from it's rolling mean, scaled by it's rolling standard deviation. The Z-score is defined as:

$$
Z_t = \frac{S_t - \mu_{S_t}}{\sigma_{S_t}}
$$

where $\mu_{S_t}$ is the spread's rolling mean over a lookback period $L$,and $\sigma_{S_t}$ is the rolling standard deviation of the spread over the same period.

In [ ]:
# function that calculate the spread (normalized)

def spread(Y, gamma):
    vect_spread = (Y.iloc[:, 0] - gamma*Y.iloc[:, 1])/(1 + np.abs(gamma))
    return vect_spread

# fonction that calculate the Z-score

def Z_score(vect_spread, lookback):
    vect_Z_score = np.zeros(vect_spread.size - lookback)

    for i in range(vect_spread.size - lookback):
        vect_Z_score[i] = (vect_spread[lookback+i] - np.mean(vect_spread[i:i+lookback]))/(np.std(vect_spread[i:i+lookback]))
    return vect_Z_score


### Générer le signal de trading

We have 
$$
s_t =
\begin{cases}
1 & \text{if } Z_t \le \text{Threshold}_{Long} \\
-1 & \text{if } Z_t \ge \text{Threshold}_{Short} \\
0 & \text{otherwise}
\end{cases}
$$

$s_t$ represent the position taken at time t (1 for long, 0 for no position and -1 for short). The thresholds are dynamically ajusted based on rolling volatility.

In [ ]:
def generate_signal(vect_Z_score, threshold_long, threshold_short):
    vect_signal = np.zeros(len(vect_Z_score))

    vect_signal[vect_Z_score <= threshold_long] = 1
    vect_signal[vect_Z_score >= threshold_short] = -1

    return vect_signal

### Compute the returns

We first consider the classical return:

$$
R_t = \text{Signal}_{t-1} \cdot \Delta S_t 
$$

where $\text{Signal}_{t-1}$ correspond to the position at $t-1$ (it can of course be no position, in this case the return is 0).

In order to have a more realistic approach, we also consider transaction cost that we applie in case of change of position (change of signal). The return write now:

$$
R_t = \text{Signal}_{t-1} \cdot \Delta S_t - c \cdot |\Delta \text{Signal}_t|
$$



In [ ]:
def compute_returns(vect_spread, signal, transaction_cost):

    spread_return = np.diff(vect_spread)

    signal_lag = np.roll(signal, 1) #because the position taken at t has impact at t+1
    signal_lag[0] = 0

    delta_signal = np.diff(signal)

    returns = signal_lag * spread_return - transaction_cost * np.abs(delta_signal)

    return returns

### Reduce de risk: Avoid trading during extreme market conditions

Especially we aim to don't buy or sell when the volatility is to high. For this purpose we first calculate the rolling volatility of spread returns using a specified lookback window $L_v$:

$$
\sigma_t = \sqrt{\frac{1}{L_v} \sum_{i=t-L_v+1}^{t} (r_i - \bar{r})^2}
$$

In [ ]:
def rolling_vol(vect_return, L_v):

    vect_vol = np.zeros(vect_return.size)

    for t in range(L_v-1, vect_return.size):

        vect_vol[t] = np.std(vect_return[t-L_v+1:t+1])
    return vect_vol

In order to decide whether we can execute order or not we compute the mean volatility (only compute on train data to avoid data leakage)

$$
\bar{\sigma} = \frac{1}{T} \sum_{t=1}^{T} \sigma_t
$$

In [ ]:
def sigma_avg(vect_vol):
    return np.mean(vect_vol)

we then define the volatility filter as the binary indicator function:

$$
\text{VolFilter}_t =
\begin{cases}
1 & \text{if } \sigma_t \le k \bar{\sigma} \\
0 & \text{if } \sigma_t > k \bar{\sigma}
\end{cases}
$$

In [1]:
def vol_filter(vect_vol, vol_avg, k):
    vect_vol_filter = vect_vol <= k*vol_avg
    return vect_vol_filter

Then we can apply this filter to our trading signals:
$$
\text{Signal}_t^{\text{filtered}} = \text{Signal}_t \cdot \text{VolFilter}_t
$$

In [ ]:
def signal_filtered(vect_signal, vect_vol_filter):
    vect_signal_filtered = vect_signal[-vect_vol_filter.size:]*vect_vol_filter
    return vect_signal_filtered

For this step, two parameters must be chosen: the lookback period $L_v$, which determines how quickly the rolling volatility adapts to market fluctuations, and the parameter $k$, which controls the level of authorized risk. These parameters will be optimized depending on the chosen assets.

## Dynamic trailing stop-loss

In [ ]:
def apply_trailing_stop(vect_spread, vect_signal, vect_vol, vol_avg, base_stop=0.025):

    n = len(vect_spread)

    signal_final = vect_signal.copy()
    position = 0

    highest = None
    lowest = None
    entry_index = None

    for t in range(n):

        
        lambda_t = base_stop * max(vect_vol[t] / vol_avg, 1)


        if position == 0 and vect_signal[t] != 0:

            position = vect_signal[t]
            entry_index = t

            highest = vect_spread[t]
            lowest = vect_spread[t]

        # position LONG

        elif position == 1:

            highest = max(highest, vect_spread[t])

            stop_level = highest * (1 - lambda_t)

            if vect_spread[t] < stop_level:

                signal_final[t] = 0
                position = 0
                highest = None
                lowest = None

        # position SHORT


        elif position == -1:

            lowest = min(lowest, spread[t])

            stop_level = lowest * (1 + lambda_t)

            if spread[t] > stop_level:

                signal_final[t] = 0
                position = 0
                highest = None
                lowest = None

    return signal_final